# 基本面綜合評分篩選 — 找出 ≥90 分的股票

**評分邏輯**（與 fundamental.html 一致，共 12 項，每項通過 +1）

| # | 欄位 | 說明 |
|---|------|------|
| 1 | 財報健康 | |
| 2 | 財報強勢 | |
| 3 | 財報動能強 | |
| 4 | EPS_前8季全為正 | |
| 5 | EPS_維持 | |
| 6 | 毛利率_維持 | |
| 7 | EPS_創新高 | |
| 8 | 毛利率_創新高 | |
| 9 | EPS_成長 | |
| 10 | EPS_YoY成長 | |
| 11 | 毛利率成長 | |
| 12 | 毛利率YoY成長 | |

分數 = pass數 / 12 × 100，≥90 分表示至少通過 11 項。

**資料來源**
- `Result/StockProfitability_all.xlsx`　→ 評分欄位取**最新一季**
- `Result/checkall/{date}.xlsx`　→ 補估值欄位（本益比、殖利率、淨值倍率、現價…）

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── 設定 ──────────────────────────────────────────────────────────
DATE         = '2026-08-03'          # ← 改這裡換日期
SCORE_CUTOFF = 70                    # ← 改這裡換門檻（0~100）

PROFIT_FILE  = Path('Result/StockProfitability_all.xlsx')
CHECKALL_FILE = Path(f'Result/checkall/{DATE}.xlsx')

# ── 評分欄位 ──────────────────────────────────────────────────────
SCORE_COLS = [
    '財報健康', '財報強勢', '財報動能強',
    'EPS_前8季全為正',
    'EPS_維持', '毛利率_維持',
    'EPS_創新高', '毛利率_創新高',
    'EPS_成長', 'EPS_YoY成長',
    '毛利率成長', '毛利率YoY成長',
]
N = len(SCORE_COLS)  # 12

print(f'日期：{DATE}　門檻：≥{SCORE_CUTOFF} 分')

日期：2026-08-03　門檻：≥70 分


In [2]:
# ── 1. 讀取財報歷史，取每支股票最新一季 ──────────────────────────
print('讀取 StockProfitability_all.xlsx ...')
df_profit = pd.read_excel(PROFIT_FILE, dtype={'stock_number': str})
df_profit['stock_number'] = df_profit['stock_number'].str.zfill(4)

# 最新一季（season_key 最大）
df_latest = (
    df_profit
    .sort_values('season_key')
    .groupby('stock_number', as_index=False)
    .last()
)

print(f'共 {len(df_latest)} 支股票，最新季範圍：{df_latest["季別"].min()} ~ {df_latest["季別"].max()}')

讀取 StockProfitability_all.xlsx ...
共 1972 支股票，最新季範圍：114.3Q ~ 115.2Q


In [3]:
# ── 2. 計算評分 ───────────────────────────────────────────────────
for col in SCORE_COLS:
    if col not in df_latest.columns:
        print(f'  ⚠️  欄位缺失：{col}，補 0')
        df_latest[col] = 0
    df_latest[col] = pd.to_numeric(df_latest[col], errors='coerce').fillna(0).astype(int)

df_latest['pass_count'] = df_latest[SCORE_COLS].sum(axis=1)
df_latest['score']      = (df_latest['pass_count'] / N * 100).round(1)

print('評分分布：')
print(df_latest['score'].describe().to_string())

評分分布：
count    1972.000000
mean       49.557657
std        27.641104
min         0.000000
25%        25.000000
50%        50.000000
75%        66.700000
max       100.000000


In [4]:
# ── 3. 讀取 checkall，補估值欄位 ─────────────────────────────────
print(f'讀取 {CHECKALL_FILE} ...')
df_check = pd.read_excel(CHECKALL_FILE, dtype={'stock_number': str})
df_check['stock_number'] = df_check['stock_number'].str.zfill(4)

# 要從 checkall 補的欄位
VAL_COLS = [
    'stock_number',
    'Type0',      # 股票名稱
    'Type1',      # 產業別
    'Type2',      # 上市/上櫃
    'now_price',  # 現價
    '_quote',     # 漲跌幅%
    '本益比', '同業平均本益比',
    '淨值倍率', '殖利率',
    '每股營收(元)',
    '營業收入-上月比較增減(%)',
    '營業收入-去年同月增減(%)',
    '累計營業收入-前期比較增減(%)',
    '總市值',
]
VAL_COLS_EXIST = [c for c in VAL_COLS if c in df_check.columns]
df_val = df_check[VAL_COLS_EXIST].copy()

print(f'checkall 共 {len(df_val)} 支股票')

讀取 Result\checkall\2026-08-03.xlsx ...
checkall 共 1520 支股票


In [5]:
# ── 4. Join ───────────────────────────────────────────────────────
df_score = df_latest[['stock_number','季別','EPS(元)','毛利率','EPS_MA_8','GM_MA_8',
                       'pass_count','score'] + SCORE_COLS].copy()

df_merged = df_score.merge(df_val, on='stock_number', how='left')

# 篩選 ≥ cutoff
df_high = df_merged[df_merged['score'] >= SCORE_CUTOFF].copy()
df_high = df_high.sort_values('score', ascending=False).reset_index(drop=True)

print(f'\n≥{SCORE_CUTOFF} 分的股票：{len(df_high)} 支')


≥70 分的股票：479 支


In [6]:
# ── 5. 展示結果 ───────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# 轉換估值欄位為數值
for col in ['本益比','同業平均本益比','淨值倍率','殖利率','now_price','EPS(元)','毛利率']:
    if col in df_high.columns:
        df_high[col] = pd.to_numeric(df_high[col], errors='coerce')

# 估值判斷
def pe_judge(row):
    pe, ipe = row.get('本益比'), row.get('同業平均本益比')
    if pd.isna(pe) or pd.isna(ipe) or ipe == 0: return '-'
    ratio = pe / ipe
    if ratio < 0.85: return '🟢 偏低'
    if ratio > 1.15: return '🔴 偏高'
    return '🟡 合理'

df_high['本益比判斷'] = df_high.apply(pe_judge, axis=1)

# 顯示欄位
SHOW = ['stock_number','Type0','Type1','score','pass_count',
        '季別','EPS(元)','毛利率',
        'now_price','本益比','同業平均本益比','本益比判斷','淨值倍率','殖利率',
        '累計營業收入-前期比較增減(%)']
SHOW = [c for c in SHOW if c in df_high.columns]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 200)

df_high[SHOW]

,stock_number,Type0,Type1,score,pass_count,季別,EPS(元),毛利率,now_price,本益比,同業平均本益比,本益比判斷,淨值倍率,殖利率,累計營業收入-前期比較增減(%)
0,1216,統一,食品工業,100.00,12,115.2Q,1.41,33.98,73.00,18.16,31.33,🟢 偏低,3.04,3.99,3.11
1,6672,騰輝電子-KY,電子零組件業,100.00,12,115.2Q,2.96,35.98,211.50,42.75,47.92,🟡 合理,3.59,1.18,26.19
2,6257,NaN,NaN,100.00,12,115.2Q,2.55,34.24,NaN,NaN,NaN,-,NaN,NaN,NaN
3,2441,超豐,半導體業,100.00,12,115.2Q,1.57,24.25,110.50,22.04,45.14,🟢 偏低,2.29,2.53,19.00
4,2451,創見,半導體業,100.00,12,115.2Q,27.70,77.03,281.00,5.05,45.14,🟢 偏低,3.15,4.06,419.72
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
474,6494,NaN,NaN,75.00,9,115.2Q,1.23,30.21,NaN,NaN,NaN,-,NaN,NaN,NaN
475,2352,佳世達,電腦及週邊設備業,75.00,9,115.2Q,0.38,17.27,28.45,37.66,26.41,🔴 偏高,1.13,3.45,2.21
476,4931,新盛力,電腦及週邊設備業,75.00,9,115.2Q,2.20,23.33,191.00,45.22,26.83,🔴 偏高,7.77,0.93,35.10
477,6526,達發,半導體業,75.00,9,115.2Q,5.25,51.16,552.00,33.56,45.14,🟢 偏低,4.90,2.23,6.96


In [7]:
# ── 6. 每項通過細節 ───────────────────────────────────────────────
print(f'\n各評分項目通過率（≥{SCORE_CUTOFF} 分的 {len(df_high)} 支）：')
for col in SCORE_COLS:
    if col in df_high.columns:
        n = df_high[col].sum()
        print(f'  {col:<18} {int(n):>4} / {len(df_high)}  ({n/len(df_high)*100:.0f}%)')


各評分項目通過率（≥70 分的 479 支）：
  財報健康                479 / 479  (100%)
  財報強勢                242 / 479  (51%)
  財報動能強               478 / 479  (100%)
  EPS_前8季全為正          285 / 479  (59%)
  EPS_維持              479 / 479  (100%)
  毛利率_維持              479 / 479  (100%)
  EPS_創新高             358 / 479  (75%)
  毛利率_創新高             328 / 479  (68%)
  EPS_成長              466 / 479  (97%)
  EPS_YoY成長           479 / 479  (100%)
  毛利率成長               429 / 479  (90%)
  毛利率YoY成長            478 / 479  (100%)


In [8]:
# ── 7. 產業分布 ───────────────────────────────────────────────────
if 'Type1' in df_high.columns:
    print(f'\n產業分布（≥{SCORE_CUTOFF} 分）：')
    print(df_high['Type1'].value_counts().to_string())


產業分布（≥70 分）：
半導體業        55
電子零組件業      48
其他電子業       25
電腦及週邊設備業    24
生技醫療業       22
鋼鐵工業        18
電機機械        17
其他          15
通信網路業       15
塑膠工業        11
化學工業        11
綠能環保        10
光電業          9
紡織纖維         9
航運業          8
建材營造         7
電子通路業        6
居家生活         5
橡膠工業         4
電器電纜         4
資訊服務業        4
食品工業         4
數位雲端         3
觀光事業         3
造紙工業         2
汽車工業         2
貿易百貨         2
運動休閒         2
農業科技業        2
油電燃氣業        2
金融業          1
玻璃陶瓷         1
金融保險         1
文化創意業        1


In [13]:
# ── 8. 輸出 Excel ─────────────────────────────────────────────────
out_path = Path(f'Result/fundamental_score_{DATE}_top{SCORE_CUTOFF}.xlsx')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path1 = Path(f'Result/fundamental_score_all.xlsx')
out_path1.parent.mkdir(parents=True, exist_ok=True)

# 輸出完整欄位
df_high.to_excel(out_path, index=False)
print(f'已輸出：{out_path}　（{len(df_high)} 支股票）')

df_merged.to_excel(out_path1, index=False)
print(f'已輸出：{out_path1}　（{len(df_merged)} 支股票）')


已輸出：Result\fundamental_score_2026-08-03_top70.xlsx　（479 支股票）
已輸出：Result\fundamental_score_all.xlsx　（1978 支股票）


In [10]:
# ── 9. 不同門檻的分布（參考用）──────────────────────────────────
print('各分數門檻對應股票數：')
for cutoff in [100, 90, 80, 70, 60, 50]:
    n = (df_merged['score'] >= cutoff).sum()
    bar = '█' * (n // 10)
    print(f'  ≥{cutoff:3d} 分：{n:4d} 支  {bar}')

各分數門檻對應股票數：
  ≥100 分： 123 支  ████████████
  ≥ 90 分： 241 支  ████████████████████████
  ≥ 80 分： 305 支  ██████████████████████████████
  ≥ 70 分： 479 支  ███████████████████████████████████████████████
  ≥ 60 分： 669 支  ██████████████████████████████████████████████████████████████████
  ≥ 50 分：1069 支  ██████████████████████████████████████████████████████████████████████████████████████████████████████████


In [11]:
print(df_high[SHOW]['stock_number'].tolist())


['1216', '6672', '6257', '2441', '2451', '6239', '6213', '6183', '6173', '2478', '2481', '6151', '2546', '6023', '6005', '5607', '5538', '6274', '2420', '6277', '6581', '2108', '6625', '2221', '2301', '6606', '6585', '2330', '6291', '6527', '6525', '6510', '6446', '2364', '2383', '2762', '5347', '5340', '3305', '3167', '4549', '4506', '3264', '3265', '3711', '3653', '4721', '3357', '3388', '3645', '3467', '3583', '3551', '4707', '3081', '5289', '4771', '5274', '5016', '5013', '4952', '3017', '3026', '3034', '4722', '3037', '3044', '3045', '4755', '4754', '4728', '6642', '6417', '1712', '1714', '7556', '7631', '7704', '7711', '7713', '7792', '7810', '7842', '8016', '8415', '8021', '1717', '1416', '6996', '6716', '8342', '8299', '8028', '1709', '8039', '8271', '8109', '8083', '1590', '1568', '1513', '7547', '1560', '6792', '6913', '6907', '9918', '9939', '8917', '6776', '2006', '2027', '6782', '6753', '6727', '2049', '1240', '2010', '2059', '8462', '6967', '2069', '6903', '8284', '9934',

In [12]:
print(df_high[df_high['EPS_前8季全為正']==1]['stock_number'].tolist())

['1216', '6672', '6257', '2441', '2451', '6239', '6213', '6183', '6173', '2478', '2481', '6151', '2546', '6023', '6005', '5607', '5538', '6274', '2420', '6277', '6581', '2108', '6625', '2221', '2301', '6606', '6585', '2330', '6291', '6527', '6525', '6510', '6446', '2364', '2383', '2762', '5347', '5340', '3305', '3167', '4549', '4506', '3264', '3265', '3711', '3653', '4721', '3357', '3388', '3645', '3467', '3583', '3551', '4707', '3081', '5289', '4771', '5274', '5016', '5013', '4952', '3017', '3026', '3034', '4722', '3037', '3044', '3045', '4755', '4754', '4728', '6642', '6417', '1712', '1714', '7556', '7631', '7704', '7711', '7713', '7792', '7810', '7842', '8016', '8415', '8021', '1717', '1416', '6996', '6716', '8342', '8299', '8028', '1709', '8039', '8271', '8109', '8083', '1590', '1568', '1513', '7547', '1560', '6792', '6913', '6907', '9918', '9939', '8917', '6776', '2006', '2027', '6782', '6753', '6727', '2049', '1240', '2010', '2059', '8462', '6967', '2069', '6903', '8438', '6435',